# Task 1 - data pipeline

Builds the unified 3-class YOLO dataset: **0 cone, 1 barrier, 2 stop_sign**.

This runs **locally**. (An earlier build was published as a Kaggle dataset, but that copy is
still awaiting compression and cannot be downloaded, so the local build is the live path.
`data/fetch_dataset.py` stays in the repo for when it is ready - put the slug back into
`configs/paths.yaml` and it works again.)

| Step | Module | Output |
| --- | --- | --- |
| Raw data | `data.fetch_nuscenes` | nuScenes v1.0-mini, ~4.2 GB, resumable |
| nuScenes -> YOLO | `data.nuscenes_to_yolo` | labels + a sidecar json per image with `K`, camera height, pitch and ground-truth depth `Z` |
| COCO stop signs | `data.coco_stopsign` | the stop-sign subset of COCO 2017, remapped to class 2 |
| Pseudo-labels | `data.pseudo_label` | stop signs that nuScenes never annotates, so the model is not taught they are background |
| Merge + checks | `data.merge`, `data.sanity_check` | `configs/data.yaml`, instance counts, 20 drawn images |

All of it in one command: `python -m data.prepare --fetch-nuscenes`. The cells below run the
same steps separately so each one's output can be looked at.

**Disk:** about 4.2 GB for the archive, 4.5 GB extracted, and very little for the dataset
itself - images are hard-linked rather than copied where the filesystem allows it.

**Downloading nuScenes means accepting its [Terms of Use](https://www.nuscenes.org/terms-of-use)**
(free for non-commercial use).

In [ ]:
# Setup. A local checkout is the normal case; the Kaggle branch is a fallback.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/heller007/eric-robotics-assessment.git"

if Path("/kaggle/working").exists():
    REPO = Path("/kaggle/working/eric-robotics-assessment")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    !pip install -q nuscenes-devkit ultralytics
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("repo:", REPO)

In [ ]:
# What this machine can do, and which sources are already here.
!python -m common.doctor

In [ ]:
# 0. The raw data (~4.2 GB). Skips itself if already extracted, and resumes if
#    interrupted. Add --dest to put it on another drive, or --archive to use a
#    .tgz you already downloaded.
!python -m data.fetch_nuscenes

In [ ]:
from common.config import load_paths
from common.env import writable_root

NUSCENES = load_paths()["nuscenes_root"]   # written by fetch_nuscenes
OUT = writable_root() / "datasets/yolo_3class"
print("nuScenes:", NUSCENES)
print("output  :", OUT)

In [ ]:
# 1. nuScenes -> YOLO. 404 samples x 6 cameras, split by scene (mini_train / mini_val):
#    a frame split would leak, since keyframes 0.5 s apart show the same cones.
!python -m data.nuscenes_to_yolo --nuscenes-root "{NUSCENES}" --out "{OUT}" --link-mode auto

In [ ]:
# 2. COCO stop signs -> class 2, into the same tree. Downloads only the ~2,000
#    images that contain one (plus the annotation zip) unless --coco-root is given.
!python -m data.coco_stopsign --out "{OUT}"

In [ ]:
# 3. Pseudo-label the stop signs nuScenes leaves unannotated. Re-running is safe:
#    previous class-2 lines on nuScenes frames are stripped first.
#    On CPU this is the slow step - lower --imgsz or pass --no-pseudo to skip it.
!python -m data.pseudo_label --out "{OUT}" --weights yolo11n.pt --conf 0.5 --imgsz 960 --batch 16

In [ ]:
# Spot-check the pseudo-labels before trusting them.
from IPython.display import Image, display

sheet = REPO / "results/pseudo_label_review.jpg"
display(Image(str(sheet))) if sheet.exists() else print("no stop signs were found on nuScenes frames")

In [ ]:
# 4. Validate the merged set and write configs/data.yaml (absolute paths, per machine).
!python -m data.merge --out "{OUT}"
!cat configs/data.yaml

In [ ]:
# 5. The checks that must pass before training: counts, per-camera K, positive Z,
#    and 20 drawn images.
!python -m data.sanity_check --out "{OUT}" --draw 20 --dest results/sanity

In [ ]:
# Eyeball the drawn boxes. Labels carry the ground-truth depth where nuScenes has one.
import itertools

for p in itertools.islice(sorted((REPO / "results/sanity/train").glob("*.jpg")), 6):
    display(Image(str(p), width=900))

## Done

Before moving on, confirm:

- both splits report non-zero cone, barrier **and** stop_sign counts;
- the drawn boxes sit on the objects (a normalisation bug shows up immediately here);
- the contact sheet really contains stop signs;
- `sanity_check` printed *all sanity checks passed*.

The dataset location is recorded in `configs/local.yaml`, which is git-ignored, so the other
machine builds its own copy with `python -m data.prepare --fetch-nuscenes`. Only code travels
through git.

Next: `task2_detection.ipynb` - B0 zero-shot baseline, then B1 / B1b / B2 fine-tuning.